# SemEval-2026 Task 13 - Task A (Google Colab Version)
## Binary Classification: Human vs Machine-Generated Code

### 🔧 Setup Instructions:
1. **Runtime**: Go to Runtime → Change runtime type → Select **GPU (T4)**
2. **Run all cells** sequentially
3. **Download results** from Files panel (left sidebar)

### 📊 Expected Results:
- Training time: ~30-60 minutes on T4 GPU
- Expected F1 Score: 70-80%
- Output: Trained model + submission.csv

## Step 1: Check GPU Availability

In [23]:
# Check if GPU is available
!nvidia-smi

Sun Mar  8 09:26:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             45W /   70W |   10433MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Install Required Libraries

In [24]:
# Install required packages
!pip install --upgrade transformers==4.45.0 datasets huggingface_hub accelerate -q

## Step 3: Import Libraries

In [25]:
import os
os.environ["WANDB_DISABLED"] = "true"
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Step 4: Define Training Class

This class handles:
- Loading data from HuggingFace
- Tokenizing code snippets
- Training CodeBERT model
- Computing metrics

In [26]:
class CodeBERTTrainer:
    def __init__(self, max_length=512, model_name="microsoft/codebert-base"):
        self.max_length = max_length
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self.num_labels = None

    def load_and_prepare_data(self):
        """Load Task A data from HuggingFace"""
        try:
            print("Loading dataset from HuggingFace...")
            dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A")

            train_data = dataset['train']
            val_data = dataset['validation']

            df = train_data.to_pandas()
            val_df = val_data.to_pandas()

            print(f"Dataset columns: {df.columns.tolist()}")
            print(f"Sample data:\n{df.head()}")

            if 'code' not in df.columns or 'label' not in df.columns:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            df = df.dropna(subset=['code', 'label'])
            val_df = val_df.dropna(subset=['code', 'label'])

            df['label'] = df['label'].astype(int)
            val_df['label'] = val_df['label'].astype(int)
            self.num_labels = df['label'].nunique()

            print(f"\nNumber of unique labels: {self.num_labels}")
            print(f"Label range: {df['label'].min()} to {df['label'].max()}")
            print(f"Label distribution:\n{df['label'].value_counts().sort_index()}")
            print(f"\nTrain samples: {len(df):,}, Validation samples: {len(val_df):,}")

            # Use subset for faster training (optional - remove these lines for full training)
            print("\n⚠️ Using 10% of data for faster demo. Remove this for competition!")
            df = df.sample(n=min(100000, len(df)), random_state=42).reset_index(drop=True)
            val_df = val_df.sample(n=min(8000, len(val_df)), random_state=42).reset_index(drop=True)
            print(f"Subset - Train: {len(df):,}, Val: {len(val_df):,}")

            return df, val_df

        except Exception as e:
            print(f"Error loading dataset: {e}")
            raise

    def initialize_model_and_tokenizer(self):
        """Initialize CodeBERT model and tokenizer"""
        print(f"\nInitializing {self.model_name} model and tokenizer...")

        self.tokenizer = RobertaTokenizer.from_pretrained(self.model_name)

        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = RobertaForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="single_label_classification"
        ).to(device)

        print(f"Model initialized with {self.num_labels} labels on {device}")

    def tokenize_function(self, examples):
        """Tokenize code snippets"""
        return self.tokenizer(
            examples['code'],
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

    def prepare_datasets(self, train_df, val_df):
        """Convert DataFrames to HuggingFace Datasets"""
        print("\nPreparing datasets for training...")

        train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
        val_dataset = Dataset.from_pandas(val_df[['code', 'label']])

        train_dataset = train_dataset.map(
            self.tokenize_function,
            batched=True,
            remove_columns=['code']
        )
        val_dataset = val_dataset.map(
            self.tokenize_function,
            batched=True,
            remove_columns=['code']
        )

        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset = val_dataset.rename_column('label', 'labels')

        return train_dataset, val_dataset

    def compute_metrics(self, eval_pred):
        """Compute evaluation metrics"""
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)

        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')

        return {
            'accuracy': accuracy,
            'f1': f1,
            'precision': precision,
            'recall': recall
        }

    def train(self, train_dataset, val_dataset, output_dir="./results", num_epochs=3, batch_size=16, learning_rate=2e-5):
        """Train the model"""
        print("\nStarting training...")

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=0.01,
            logging_dir='./logs',
            logging_steps=100,
            evaluation_strategy="steps",
            eval_steps=500, # changed from 500
            save_strategy="steps",
            save_steps=500, # changed from 500
            dataloader_num_workers=2,        # ← ADD: Parallel data loading
            dataloader_pin_memory=True,      # ← ADD: Faster GPU transfer
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="linear",
            save_total_limit=2,
            report_to="none",
            fp16=True # New added
        )

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )

        trainer.train()

        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)

        print(f"\n✅ Training completed. Model saved to {output_dir}")

        return trainer

    def evaluate_model(self, trainer, val_dataset):
        """Evaluate the trained model"""
        print("\n📊 Evaluating model...")

        predictions = trainer.predict(val_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids

        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, target_names=['Human', 'Machine']))

        return predictions

## Step 5: Load Data

⏰ This may take 2-5 minutes to download from HuggingFace

In [27]:
# Initialize trainer
trainer = CodeBERTTrainer(max_length=256, model_name="microsoft/codebert-base")

# Load data
train_df, val_df = trainer.load_and_prepare_data()

Loading dataset from HuggingFace...
Dataset columns: ['code', 'generator', 'label', 'language']
Sample data:
                                                code  \
0  (a, b, c, d) = [int(x) for x in input().split(...   
1  valid version for the language; all others can...   
2  python\ndef min_cards_to_flip(s):\n    vowels ...   
3  T = int(input())\nfor t in range(T):\n\tcolor ...   
4  for i in range(int(input())):\n\tinput()\n\ta ...   

                        generator  label language  
0                           human      0   Python  
1         Qwen/Qwen2.5-Coder-1.5B      1   Python  
2  Qwen/Qwen2.5-Coder-7B-Instruct      1   Python  
3                           human      0   Python  
4                           human      0   Python  

Number of unique labels: 2
Label range: 0 to 1
Label distribution:
label
0    238475
1    261525
Name: count, dtype: int64

Train samples: 500,000, Validation samples: 100,000

⚠️ Using 10% of data for faster demo. Remove this for competitio

## Step 6: Initialize Model

In [28]:
# Initialize model and tokenizer
trainer.initialize_model_and_tokenizer()


Initializing microsoft/codebert-base model and tokenizer...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model initialized with 2 labels on cuda


## Step 7: Prepare Datasets

In [29]:
# Prepare datasets
train_dataset, val_dataset = trainer.prepare_datasets(train_df, val_df)

print(f"\nTrain dataset: {len(train_dataset)} samples")
print(f"Validation dataset: {len(val_dataset)} samples")


Preparing datasets for training...


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]


Train dataset: 100000 samples
Validation dataset: 8000 samples


## Step 8: Train the Model

⏰ **Training Time**:
- With subset (50K samples): ~15-30 minutes
- Full dataset (500K samples): ~2-3 hours

💡 **Tip**: You can adjust hyperparameters below:
- `num_epochs`: More epochs = better performance (but longer training)
- `batch_size`: Larger = faster but needs more GPU memory
- `learning_rate`: Try 1e-5 to 5e-5

In [30]:
# Train the model
trained_model = trainer.train(
    train_dataset,
    val_dataset,
    output_dir="./task_a_model",
    num_epochs=3,
    batch_size=64,
    learning_rate=2e-5
)


Starting training...


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
500,0.065300,0.079937,0.977125,0.977113,0.976999,0.977645
1000,0.053400,0.054818,0.983250,0.983232,0.983102,0.983406
1500,0.044300,0.043563,0.985750,0.985731,0.985696,0.985767
2000,0.028300,0.046362,0.986625,0.986610,0.986482,0.986781
2500,0.033200,0.040116,0.987750,0.987737,0.987602,0.987922
3000,0.028900,0.044583,0.987500,0.987488,0.987331,0.987728
3500,0.016600,0.041976,0.989250,0.989237,0.989150,0.989339
4000,0.019500,0.039467,0.989875,0.989863,0.989761,0.989988
4500,0.019400,0.040359,0.989875,0.989863,0.989755,0.989998



✅ Training completed. Model saved to ./task_a_model


## Step 9: Evaluate the Model

In [35]:
# Evaluate on validation set
import json
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, f1_score, precision_score, recall_score, confusion_matrix

# Load the full validation dataset to get 20K samples
print("Loading full validation dataset for evaluation...")
full_val_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A")['validation']
full_val_df = full_val_dataset.to_pandas()

# Select random 20K samples for evaluation
eval_sample_size = min(20000, len(full_val_df))
eval_df = full_val_df.sample(n=eval_sample_size, random_state=42).reset_index(drop=True)
print(f"\n📊 Evaluating on {len(eval_df):,} random validation samples")

# Prepare evaluation dataset
eval_dataset = Dataset.from_pandas(eval_df[['code', 'label']])
eval_dataset = eval_dataset.map(
    trainer.tokenize_function,
    batched=True,
    remove_columns=['code']
)
eval_dataset = eval_dataset.rename_column('label', 'labels')

# Run evaluation
print("\n🔄 Running evaluation (this may take 2-3 minutes)...")
predictions = trained_model.predict(eval_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

# Compute detailed metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, labels=[0, 1]
)
macro_f1 = f1_score(y_true, y_pred, average='macro')
macro_precision = precision_score(y_true, y_pred, average='macro')
macro_recall = recall_score(y_true, y_pred, average='macro')

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Print results
print("\n" + "="*60)
print("📊 EVALUATION RESULTS ON 20K RANDOM SAMPLES")
print("="*60)
print(f"\n🎯 Overall Metrics:")
print(f"   Accuracy:        {accuracy:.4f}")
print(f"   Macro F1:        {macro_f1:.4f} ⭐")
print(f"   Macro Precision: {macro_precision:.4f}")
print(f"   Macro Recall:    {macro_recall:.4f}")

print(f"\n📈 Per-Class Metrics:")
print(f"   Class 0 (Human):")
print(f"      Precision: {precision[0]:.4f}")
print(f"      Recall:    {recall[0]:.4f}")
print(f"      F1-Score:  {f1[0]:.4f}")
print(f"      Support:   {support[0]}")
print(f"\n   Class 1 (Machine):")
print(f"      Precision: {precision[1]:.4f}")
print(f"      Recall:    {recall[1]:.4f}")
print(f"      F1-Score:  {f1[1]:.4f}")
print(f"      Support:   {support[1]}")

print(f"\n🔢 Confusion Matrix:")
print(f"                Predicted")
print(f"                Human  Machine")
print(f"   Actual Human   {cm[0][0]:5d}  {cm[0][1]:5d}")
print(f"   Actual Machine {cm[1][0]:5d}  {cm[1][1]:5d}")

# Detailed classification report
print("\n📋 Detailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Human', 'Machine'], digits=4))

# Save results to files
results_dict = {
    'evaluation_samples': eval_sample_size,
    'overall_metrics': {
        'accuracy': float(accuracy),
        'macro_f1': float(macro_f1),
        'macro_precision': float(macro_precision),
        'macro_recall': float(macro_recall)
    },
    'per_class_metrics': {
        'human': {
            'precision': float(precision[0]),
            'recall': float(recall[0]),
            'f1_score': float(f1[0]),
            'support': int(support[0])
        },
        'machine': {
            'precision': float(precision[1]),
            'recall': float(recall[1]),
            'f1_score': float(f1[1]),
            'support': int(support[1])
        }
    },
    'confusion_matrix': {
        'true_human_pred_human': int(cm[0][0]),
        'true_human_pred_machine': int(cm[0][1]),
        'true_machine_pred_human': int(cm[1][0]),
        'true_machine_pred_machine': int(cm[1][1])
    }
}

# Save as JSON
with open('evaluation_results.json', 'w') as f:
    json.dump(results_dict, f, indent=2)
print("\n✅ Results saved to: evaluation_results.json")

# Save detailed results as CSV
results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Macro F1', 'Macro Precision', 'Macro Recall',
               'Human Precision', 'Human Recall', 'Human F1',
               'Machine Precision', 'Machine Recall', 'Machine F1'],
    'Value': [accuracy, macro_f1, macro_precision, macro_recall,
              precision[0], recall[0], f1[0],
              precision[1], recall[1], f1[1]]
})
results_df.to_csv('evaluation_metrics.csv', index=False)
print("✅ Metrics saved to: evaluation_metrics.csv")

# Save predictions with true labels for error analysis
eval_results_df = pd.DataFrame({
    'sample_id': range(len(y_true)),
    'true_label': y_true,
    'predicted_label': y_pred,
    'correct': y_true == y_pred,
    'true_label_name': ['Human' if x == 0 else 'Machine' for x in y_true],
    'predicted_label_name': ['Human' if x == 0 else 'Machine' for x in y_pred]
})
eval_results_df.to_csv('evaluation_predictions.csv', index=False)
print("✅ Detailed predictions saved to: evaluation_predictions.csv")

print("\n" + "="*60)
print("📥 Files ready for download:")
print("   1. evaluation_results.json - Summary metrics (JSON)")
print("   2. evaluation_metrics.csv - Metrics table (CSV)")
print("   3. evaluation_predictions.csv - All predictions (CSV)")
print("="*60)

Loading full validation dataset for evaluation...

📊 Evaluating on 20,000 random validation samples


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]


🔄 Running evaluation (this may take 2-3 minutes)...



📊 EVALUATION RESULTS ON 20K RANDOM SAMPLES

🎯 Overall Metrics:
   Accuracy:        0.9913
   Macro F1:        0.9913 ⭐
   Macro Precision: 0.9912
   Macro Recall:    0.9914

📈 Per-Class Metrics:
   Class 0 (Human):
      Precision: 0.9881
      Recall:    0.9938
      F1-Score:  0.9909
      Support:   9575

   Class 1 (Machine):
      Precision: 0.9943
      Recall:    0.9890
      F1-Score:  0.9916
      Support:   10425

🔢 Confusion Matrix:
                Predicted
                Human  Machine
   Actual Human    9516     59
   Actual Machine   115  10310

📋 Detailed Classification Report:
              precision    recall  f1-score   support

       Human     0.9881    0.9938    0.9909      9575
     Machine     0.9943    0.9890    0.9916     10425

    accuracy                         0.9913     20000
   macro avg     0.9912    0.9914    0.9913     20000
weighted avg     0.9913    0.9913    0.9913     20000


✅ Results saved to: evaluation_results.json
✅ Metrics saved to: evalu

## Step 10: Generate Predictions (Optional)

If you have test data, you can generate predictions for submission

In [32]:
# Generate predictions on validation set as example
predictions = trained_model.predict(val_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

# Create submission DataFrame
submission_df = pd.DataFrame({
    'ID': range(len(pred_labels)),
    'prediction': pred_labels
})

# Save to CSV
submission_df.to_csv('task_a_submission.csv', index=False)
print("\n✅ Predictions saved to: task_a_submission.csv")
print(f"\nFirst 10 predictions:")
print(submission_df.head(10))


✅ Predictions saved to: task_a_submission.csv

First 10 predictions:
   ID  prediction
0   0           0
1   1           0
2   2           1
3   3           0
4   4           0
5   5           1
6   6           0
7   7           1
8   8           1
9   9           1


## Step 11: Download Results

### 📥 Files to Download:
1. **`task_a_model/`** - Trained model folder (for later use)
2. **`task_a_submission.csv`** - Predictions file

### How to Download:
1. Click the **Files** icon (📁) in the left sidebar
2. Right-click on files/folders
3. Select **Download**

### Or use code:

In [33]:
# Compress model folder for easier download
!zip -r task_a_model.zip task_a_model/
print("✅ Model compressed to: task_a_model.zip")
print("\n📥 Download these files from the Files panel:")
print("   - task_a_model.zip")
print("   - task_a_submission.csv")

  adding: task_a_model/ (stored 0%)
  adding: task_a_model/special_tokens_map.json (deflated 84%)
  adding: task_a_model/model.safetensors (deflated 7%)
  adding: task_a_model/tokenizer_config.json (deflated 76%)
  adding: task_a_model/training_args.bin (deflated 53%)
  adding: task_a_model/checkpoint-4689/ (stored 0%)
  adding: task_a_model/checkpoint-4689/special_tokens_map.json (deflated 84%)
  adding: task_a_model/checkpoint-4689/trainer_state.json (deflated 75%)
  adding: task_a_model/checkpoint-4689/model.safetensors (deflated 7%)
  adding: task_a_model/checkpoint-4689/scheduler.pt (deflated 61%)
  adding: task_a_model/checkpoint-4689/tokenizer_config.json (deflated 76%)
  adding: task_a_model/checkpoint-4689/training_args.bin (deflated 53%)
  adding: task_a_model/checkpoint-4689/optimizer.pt (deflated 21%)
  adding: task_a_model/checkpoint-4689/config.json (deflated 50%)
  adding: task_a_model/checkpoint-4689/rng_state.pth (deflated 26%)
  adding: task_a_model/checkpoint-4689/me

## 🎉 Congratulations!

You've successfully trained a baseline model for Task A!

### 📊 Next Steps:
1. **Improve Performance**:
   - Remove the data subset (use full 500K samples)
   - Train for more epochs (5-10)
   - Try different models: `microsoft/graphcodebert-base`, `Salesforce/codet5-base`
   - Increase `max_length` to 512

2. **Create Real Submissions**:
   - Load test data (when available)
   - Generate predictions
   - Submit to competition

3. **Experiment**:
   - Try ensemble methods
   - Add feature engineering
   - Fine-tune hyperparameters

### 📚 Resources:
- [Competition Page](https://groups.google.com/g/semeval2026-task13)
- [Dataset on HuggingFace](https://huggingface.co/datasets/DaniilOr/SemEval-2026-Task13)
- [CodeBERT Paper](https://arxiv.org/abs/2002.08155)